# MLB Offensive Production Analysis

This notebook studies same-season associations between hitter traits and offensive production for model-eligible player-seasons in the cached 2024–2026 dataset. It is not a causal analysis or a next-season forecast.

The analysis has three acts. This pass adds **Act 1**: rank swing and stance traits by how strongly they correlate with production.

- **Outcome:** FanGraphs **wRC+**, with Statcast **xwOBA** as a second check (results vs expected contact).
- **Traits:** bat tracking and stance/setup (bat speed, attack angle, distance off the plate, and related Statcast fields).
- **Excluded from the ranking:** barrels, exit velocity, OPS, and wOBA. Those are production itself or batted-ball results sitting downstream of the swing; putting them on the list would crowd out the traits we actually want to rank.

Acts 2–3 (overlap with core stats, then reconstruction) remain later in the notebook.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from mlb_offense.analysis import (
    AnalysisConfig,
    bootstrap_model_difference,
    clustered_ols_coefficients,
    coverage_by_season,
    evaluate_models,
    feature_sets,
    format_trait_ranking,
    grouped_permutation_importance,
    load_analysis_data,
    missingness_by_season,
    plot_missingness,
    plot_outcome_distributions,
    plot_predictions,
    plot_residual_diagnostics,
    plot_trait_ranking,
    rank_hitter_traits,
)

DATA_PATH = Path('data/processed/mlb_offense_2024_2026.parquet')
OUTPUT_DIR = Path('data/analysis')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

config = AnalysisConfig(
    min_pa=100,
    min_competitive_swings=50,
    outer_splits=5,
    inner_splits=3,
    bootstrap_iterations=500,
)
data = load_analysis_data(DATA_PATH, config)
blocks = feature_sets(data)
print(f'{len(data):,} eligible player-seasons across {data.player_id.nunique():,} hitters')

## Sample coverage and descriptive outcomes

The local thresholds are intentionally applied after data collection. This keeps the raw source responses intact and makes sensitivity checks straightforward.

In [ ]:
display(coverage_by_season(data).style.format(precision=3))
display(
    data.groupby('season')[['pa', 'competitive_swings', 'wrc_plus', 'ops_plus', 'woba', 'xwoba']]
    .describe()
)

## Act 1: Which swing and stance traits track production?

The ranking below is the simple question: among **how a hitter swings and sets up**, which fields move with offensive production?

Each row is an eligible player-season (`PA >= 100`, `competitive swings >= 50`). Pearson *r* is linear association; Spearman *ρ* is the rank-based check. **wRC+** is observed run production. **xwOBA** is expected production from contact quality, so a trait that ranks high on both is not just riding BABIP noise.

### Why barrels, exit velocity, OPS, and wOBA are not on this list

Those four would dominate a correlation table, and they would do it for the wrong reason.

- **OPS and wOBA are production.** OPS is a counting-rate mix of OBP and SLG. wOBA is a linear-weights production metric. wRC+ is a scaled run-production index built from the same family of ingredients. Asking how much OPS or wOBA correlates with wRC+ is nearly tautological: you would be ranking the outcome against itself. They belong as *targets*, not as traits. xwOBA is used here only as a second outcome, never as a trait.
- **Barrels and exit velocity are batted-ball results, not swing traits.** A barrel is already a ball hit in a high-value exit-velo / launch-angle window. Average EV is how hard the ball came off the bat. Both sit *downstream* of the swing. If they were included, they would crowd out bat speed, attack angle, and stance because they are closer to the production mechanism. They return later as part of the `core` block, when the question becomes whether swing traits add anything *after* contact quality is known.

Hard-hit rate and average launch angle are kept off the ranking for the same downstream-contact reason. Plate-discipline rates (walks, strikeouts, chase) are also not swing/stance traits; they are approach outcomes and belong with the core stats.

What remains is the list that matches the original goal: bat speed, swing length, squared-up and blast rates, whiff, attack angle and direction, tilt, distance off the plate, depth in the box, and intercept.

In [ ]:
ranking = rank_hitter_traits(data)
ranking.to_csv(OUTPUT_DIR / 'trait_outcome_rankings.csv', index=False)

ranking_table = format_trait_ranking(ranking)
ranking_table.to_csv(OUTPUT_DIR / 'trait_ranking_wide.csv', index=False)
display(
    ranking_table.style.format({
        column: '{:.3f}' if 'n_' not in column else '{:.0f}'
        for column in ranking_table.columns
        if column not in {'trait', 'trait_label'}
    })
)

ranking_figure = plot_trait_ranking(ranking)
ranking_figure.savefig(OUTPUT_DIR / 'trait_ranking.png', dpi=160, bbox_inches='tight')
plt.show()

The ranking is descriptive association among eligible player-seasons, not a claim that any trait causes production.

Blast rate (Statcast’s combination of a fast swing and a squared-up contact) leads both lists. Fast-swing rate and average bat speed follow. Stance and intercept fields sit near zero. Correlations with xwOBA are stronger than with wRC+ for the top traits, which is what you would expect if those traits track contact quality more tightly than realized results.

Squared-up and blast rates are still *swing* descriptors, not batted-ball outcomes like barrels or EV. They stay in Act 1 for that reason. They will overlap with the core contact block in Act 2.

In [ ]:
figures = [
    plot_outcome_distributions(data),
    plot_missingness(data),
]
for number, figure in enumerate(figures, start=1):
    figure.savefig(OUTPUT_DIR / f'eda_{number}.png', dpi=160, bbox_inches='tight')
    plt.show()

display(missingness_by_season(data).sort_values(['season', 'missing_rate'], ascending=[True, False]).head(20))

## Leakage-safe grouped model evaluation

Predictors are evaluated in two prespecified blocks: `core` contains age, handedness, season, plate discipline, and contact-quality measures; `core_plus_traits` adds bat-tracking and stance fields. Production metrics and their components are excluded. Hyperparameters are tuned inside each training fold; the outer evaluation folds are grouped by hitter.

In [ ]:
performance, predictions, selected_parameters = evaluate_models(data, config)
performance.to_csv(OUTPUT_DIR / 'grouped_cv_performance.csv', index=False)
predictions.to_parquet(OUTPUT_DIR / 'out_of_fold_predictions.parquet', index=False)
selected_parameters.to_json(OUTPUT_DIR / 'selected_parameters.json', orient='records', indent=2)

display(
    performance.style.format({
        'mae': '{:.2f}', 'rmse': '{:.2f}', 'r2': '{:.3f}',
        'calibration_intercept': '{:.2f}', 'calibration_slope': '{:.3f}',
        'pa_weighted_mae': '{:.2f}', 'pa_weighted_rmse': '{:.2f}',
        'pa_weighted_r2': '{:.3f}',
    })
)

In [ ]:
comparison = bootstrap_model_difference(
    predictions,
    candidate_model='elastic_net:core_plus_traits',
    reference_model='elastic_net:core',
    metric='mae',
    iterations=config.bootstrap_iterations,
    random_state=config.random_state,
)
comparison.to_csv(OUTPUT_DIR / 'elastic_net_trait_increment_bootstrap.csv', index=False)
display(comparison.style.format({'difference': '{:.3f}', 'ci_lower': '{:.3f}', 'ci_upper': '{:.3f}'}))

best_model = performance.iloc[0]['model']
prediction_figure = plot_predictions(predictions, best_model)
prediction_figure.savefig(OUTPUT_DIR / 'best_model_calibration.png', dpi=160, bbox_inches='tight')
plt.show()

residual_figure = plot_residual_diagnostics(predictions, best_model, data)
residual_figure.savefig(OUTPUT_DIR / 'best_model_residuals.png', dpi=160, bbox_inches='tight')
plt.show()

## Interpretation aids

The standardized OLS coefficients below use player-clustered standard errors. Permutation importance is descriptive: correlated hitter traits can substitute for each other and are not independent causal effects.

In [ ]:
coefficients = clustered_ols_coefficients(data, blocks['core_plus_traits'], config)
coefficients.to_csv(OUTPUT_DIR / 'clustered_ols_coefficients.csv', index=False)
display(coefficients.head(20).style.format({
    'coefficient': '{:.3f}', 'std_error': '{:.3f}', 'ci_lower': '{:.3f}',
    'ci_upper': '{:.3f}', 'p_value': '{:.4f}',
}))

importance = grouped_permutation_importance(data, blocks['core_plus_traits'], config)
importance.to_csv(OUTPUT_DIR / 'permutation_importance.csv', index=False)
display(importance.head(20).style.format({'importance_mean': '{:.3f}', 'importance_std': '{:.3f}'}))

## Interpretation checklist

- The table compares observed production in the same season, so it should be interpreted as association rather than prediction or causation.
- Compare `elastic_net:core_plus_traits` against `elastic_net:core` and inspect the bootstrapped MAE difference to assess incremental value from bat and stance traits.
- Check residual plots and calibration before trusting aggregate error statistics.
- Re-run with alternative PA and competitive-swing thresholds before making conclusions about the MLB hitter population.